# Reading From Bronze

In the silver layer, we will only clean the data for each table. (Merging / data modeling will be done in the Gold Layer)

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

# Data Transformation

Data catelog description from OurAirports:
| FIELD NAMES | DESCRIPTION |
| ----------- | ----------- |
| id | Internal OurAirports integer identifier for the airport. This will stay persistent, even if the airport code changes. |
| ident | The text identifier used in the OurAirports URL. This will be the ICAO code if available. Otherwise, it will be a local airport code (if no conflict), or if nothing else is available, an internally-generated code starting with the ISO2 country code, followed by a dash and a four-digit number. |
| type | 	The type of the airport. Allowed values are "balloonport", "closed_airport", "heliport", "large_airport", "medium_airport", "seaplane_base", and "small_airport". |
| name | The official airport name, including "Airport", "Airstrip", etc. |
| latitude_deg | The airport latitude in decimal degrees (positive for north). |
| longitude_deg | The airport longitude in decimal degrees (positive for east). |
| elevation_ft | The airport elevation MSL in feet (not metres). |
| continent | The code for the continent where the airport is (primarily) located. Allowed values are "AF" (Africa), "AN" (Antarctica), "AS" (Asia), "EU" (Europe), "NA" (North America), "OC" (Oceania), or "SA" (South America). |
| iso_country | The two-character ISO 3166:1-alpha2 code for the country where the airport is (primarily) located. A handful of unofficial, non-ISO codes are also in use, such as "XK" for Kosovo. |
| iso_region | An alphanumeric code for the high-level administrative subdivision of a country where the airport is primarily located (e.g. province, governorate), prefixed by the ISO2 country code and a hyphen. OurAirports uses ISO 3166:2 codes whenever possible, preferring higher administrative levels, but also includes some custom codes.  |
| municipality | The primary municipality that the airport serves (when available). Note that this is not necessarily the municipality where the airport is physically located. |
| scheduled_service | "yes" if the airport currently has scheduled airline service; "no" otherwise. |
| gps_code | The code that an aviation GPS database (such as Jeppesen's or Garmin's) would normally use for the airport. This will always be the icao_code if the airport has one, but it is not guaranteed to be globally unique. |
| icao_code | The four-letter ICAO code for the airport (if it has one). |
| iata_code | The three-letter IATA code for the airport (if it has one). |
| local_code | The local country code for the airport, if different from the gps_code, icao_code, and iata_code fields (used mainly for US airports). |
| home_link | URL of the airport's official home page on the web, if one exists. |
| wikipedia_link | URL of the airport's page on Wikipedia, if one exists. |
| keywords | 	Extra keywords/phrases to assist with search, comma-separated. May include former names for the airport, alternate codes, names in other languages, nearby tourist destinations, etc. |


In [0]:
df_airports = spark.table("bronze_travel.airports")

## Exploring raw data

In [0]:
df_airports.display()

In [0]:
# Check data types

df_airports.dtypes

**Clean Up To Do List:**

- Trim white spaces on every column
- Remove duplicates
- key columns cannot be null: "name"

## Trim white spaces from all columns

In [0]:
for column_name, data_type in df_airports.dtypes:
  if data_type == "string":
    df_airports = df_airports.withColumn(column_name, F.trim(F.col(column_name)))

## Drop Duplicates

In [0]:
# Find groups with more than one row
dup_groups = (
    df_airports.groupBy(*df_airports.columns)
    .count()
    .filter(F.col("count") > 1)
)

# Number of *extra* rows that dropDuplicates will remove
# (count - 1 per group, summed across all groups)
dup_row_count = (
    dup_groups
    .withColumn("extra_rows", F.col("count") - 1)
    .agg(F.sum("extra_rows"))
    .first()[0]
) or 0  # guard against None if there are no duplicates at all

print(f"Found {dup_row_count} duplicate rows across {dup_groups.count()} groups")

before_count = df_airports.count()

# Drop duplicates, keeping one row per group
df_airports = df_airports.dropDuplicates(df_airports.columns)

after_count = df_airports.count()
actual_dropped = before_count - after_count

print(f"Before: {before_count}, After: {after_count}, Dropped: {actual_dropped}")

if actual_dropped != dup_row_count:
    raise ValueError(
        f"Expected to drop {dup_row_count} duplicate rows but dropped {actual_dropped}. "
    )

## Check for NULL Values

In [0]:
null_count = df_airports.select(F.sum(F.col("name").isNull().cast("int")).alias("count")).first()["count"]

if null_count > 0:

    print(f"Found {null_count} NULLs in 'name' column. Dropping NULL rows.")
    df_airports = df_airports.filter(F.col("name").isNotNull())
else:
    print(f"Found {null_count} nulls in 'name' column")

## Update iso_region column
Remove country prefix.

In [0]:
df_airports = df_airports.withColumn("iso_region", F.regexp_replace("iso_region", r"^[^-]*-", ""))

# Write Into Silver Table

In [0]:
df_airports.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("silver_travel.airports")

In [0]:
%sql
SELECT *
FROM silver_travel.airports
LIMIT 100